# Distribution Shift Analysis

This notebook characterises the distribution shift between GUIDE-seq and CHANGE-seq using the preprocessed datasets and DNABERT deep ensemble.

The analysis separates three related effects: guide familiarity, assay-dependent differences in observed labels, and changes in model behaviour under cross-assay evaluation. Exact genomic sites measured in both assays are compared to determine whether observed labels differ even when guide and candidate-site composition are controlled.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss)

In [2]:
DATA_DIR = 'data/processed/'
GRNA_COL = 'target'
SITE_KEY = [GRNA_COL, 'chrom', 'chromStart', 'strand']

guide_train = pd.read_csv(DATA_DIR + 'guide_train.csv', low_memory=False)
guide_test = pd.read_csv(DATA_DIR + 'guide_test.csv', low_memory=False)
change_train = pd.read_csv(DATA_DIR + 'change_train.csv', low_memory=False)
change_test = pd.read_csv(DATA_DIR + 'change_test.csv', low_memory=False)

guide_full = pd.concat([guide_train, guide_test], ignore_index=True)
change_full = pd.concat([change_train, change_test], ignore_index=True)

# Final preprocessing should contain one row per genomic site.
assert guide_full.duplicated(subset=SITE_KEY).sum() == 0
assert change_full.duplicated(subset=SITE_KEY).sum() == 0

# Within each assay, train and test guides must remain disjoint.
assert set(guide_train[GRNA_COL]).isdisjoint(set(guide_test[GRNA_COL]))
assert set(change_train[GRNA_COL]).isdisjoint(set(change_test[GRNA_COL]))

print(
    f'GUIDE-seq full: {len(guide_full):,} rows, '
    f'{guide_full[GRNA_COL].nunique()} guides'
)
print(
    f'CHANGE-seq full: {len(change_full):,} rows, '
    f'{change_full[GRNA_COL].nunique()} guides'
)

print(f'\nGUIDE train: {len(guide_train):,}')
print(f'GUIDE test: {len(guide_test):,}')
print(f'CHANGE train: {len(change_train):,}')
print(f'CHANGE test: {len(change_test):,}')

GUIDE-seq full: 1,477,762 rows, 58 guides
CHANGE-seq full: 2,873,627 rows, 110 guides

GUIDE train: 1,229,148
GUIDE test: 248,614
CHANGE train: 2,352,636
CHANGE test: 520,991


In [3]:
y_guide_test = guide_test['label'].to_numpy()
y_change_test = change_test['label'].to_numpy()

guide_logits = np.load(DATA_DIR + 'guide_member_logits.npy')
change_logits = np.load(DATA_DIR + 'change_member_logits.npy')

metadata = np.load(DATA_DIR + 'ensemble_training_metadata.npz')

negative_sampling_rate = float(metadata['negative_sampling_rate'])
OFFSET = float(metadata['sampling_logit_offset'])

# Checking saved predictions still align with the final test sets.
assert guide_logits.shape == (5, len(y_guide_test))
assert change_logits.shape == (5, len(y_change_test))

assert np.isfinite(guide_logits).all()
assert np.isfinite(change_logits).all()

# Applies the training-prior correction to each member before averaging.
guide_member_probs = expit(guide_logits + OFFSET)
change_member_probs = expit(change_logits + OFFSET)

guide_mean_probs = guide_member_probs.mean(axis=0)
change_mean_probs = change_member_probs.mean(axis=0)

# Ensemble disagreement as the epistemic uncertainty proxy.
guide_uncertainty = guide_member_probs.var(axis=0)
change_uncertainty = change_member_probs.var(axis=0)

print(f'GUIDE logits: {guide_logits.shape}')
print(f'CHANGE logits: {change_logits.shape}')

print(f'\nNegative sampling rate: {negative_sampling_rate:.8f}')
print(f'Logit correction offset: {OFFSET:.6f}')

GUIDE logits: (5, 248614)
CHANGE logits: (5, 520991)

Negative sampling rate: 0.01116652
Logit correction offset: -4.494835


In [4]:
def ece(y, p, n_bins=10, strategy='quantile'):
    if strategy == 'quantile':
        edges = np.unique(
            np.quantile(p, np.linspace(0, 1, n_bins + 1))
        )
    else:
        edges = np.linspace(0, 1, n_bins + 1)

    idx = np.clip(
        np.digitize(p, edges[1:-1], right=False),
        0,
        len(edges) - 2
    )

    total = 0.0
    for b in range(len(edges) - 1):
        mask = idx == b
        if not mask.any():
            continue
        total += (
            mask.mean()
            * abs(p[mask].mean() - y[mask].mean())
        )
    return total

## Guide Overlap Between Assays

GUIDE-seq and CHANGE-seq contain overlapping gRNAs. Full-dataset guide overlap is examined first to establish how much of the cross-assay comparison involves the same guide sequences before evaluating the model on subsets.

In [6]:
guide_guides = set(guide_full[GRNA_COL].unique())
change_guides = set(change_full[GRNA_COL].unique())

shared_guides = guide_guides & change_guides

guide_shared_rows = guide_full[GRNA_COL].isin(shared_guides).mean()
change_shared_rows = change_full[GRNA_COL].isin(shared_guides).mean()

print('Full-dataset guide overlap:')
print(f'GUIDE-seq guides: {len(guide_guides)}')
print(f'CHANGE-seq guides: {len(change_guides)}')
print(f'Shared guides: {len(shared_guides)}')
print(f'GUIDE-only guides: {len(guide_guides - change_guides)}')
print(f'CHANGE-only guides: {len(change_guides - guide_guides)}')

print(
    f'\nShared guides represent '
    f'{100 * guide_shared_rows:.1f}% of GUIDE rows'
)
print(
    f'Shared guides represent '
    f'{100 * change_shared_rows:.1f}% of CHANGE rows'
)

Full-dataset guide overlap:
GUIDE-seq guides: 58
CHANGE-seq guides: 110
Shared guides: 58
GUIDE-only guides: 0
CHANGE-only guides: 52

Shared guides represent 100.0% of GUIDE rows
Shared guides represent 46.0% of CHANGE rows


### Results

Although the GUIDE-seq and CHANGE-seq splits are independently gRNA-disjoint within each assay, some gRNAs in the CHANGE-seq test set also occur in the GUIDE-seq training set.

Because CHANGE-seq represents a different experimental assay, these examples are not treated as conventional same-dataset train-test leakage. Instead, the CHANGE-seq test set is divided into guide-familiar and guide-novel subsets so that cross-assay transfer can be assessed separately from guide familiarity.

In [8]:
guide_train_guides = set(guide_train[GRNA_COL].unique())
change_test_guides = set(change_test[GRNA_COL].unique())

familiar_guides = guide_train_guides & change_test_guides
novel_guides = change_test_guides - guide_train_guides

familiar_mask = change_test[GRNA_COL].isin(familiar_guides).to_numpy()
novel_mask = ~familiar_mask

assert familiar_mask.sum() + novel_mask.sum() == len(change_test)

print('GUIDE-train / CHANGE-test guide familiarity:')
print(f'GUIDE train guides: {len(guide_train_guides)}')
print(f'CHANGE test guides: {len(change_test_guides)}')
print(f'Guide-familiar CHANGE guides: {len(familiar_guides)}')
print(f'Guide-novel CHANGE guides: {len(novel_guides)}')

print(
    f'\nGuide-familiar rows: '
    f'{familiar_mask.sum():,} / {len(change_test):,} '
    f'({100 * familiar_mask.mean():.1f}%)'
)

print(
    f'Guide-novel rows: '
    f'{novel_mask.sum():,} / {len(change_test):,} '
    f'({100 * novel_mask.mean():.1f}%)'
)

print(
    f'\nGuide-familiar positives: '
    f'{y_change_test[familiar_mask].sum():,} '
    f'({y_change_test[familiar_mask].mean():.4%})'
)

print(
    f'Guide-novel positives: '
    f'{y_change_test[novel_mask].sum():,} '
    f'({y_change_test[novel_mask].mean():.4%})'
)

GUIDE-train / CHANGE-test guide familiarity:
GUIDE train guides: 46
CHANGE test guides: 22
Guide-familiar CHANGE guides: 12
Guide-novel CHANGE guides: 10

Guide-familiar rows: 248,375 / 520,991 (47.7%)
Guide-novel rows: 272,616 / 520,991 (52.3%)

Guide-familiar positives: 2,528 (1.0178%)
Guide-novel positives: 4,082 (1.4973%)


## Model Behaviour by Shift Condition

The pooled CHANGE-seq test result combines two different cross-assay conditions. To separate assay transfer from guide familiarity, model behaviour is compared across three conditions:

1. novel guide, same assay — held-out GUIDE-seq test guides;
2. familiar guide, cross assay — CHANGE-seq test guides present during GUIDE-seq training;
3. novel guide, cross assay — CHANGE-seq test guides absent from GUIDE-seq training.

Discrimination, calibration and ensemble disagreement are reported for each condition.

In [9]:
conditions = [
    (
        'Novel guide, same assay',
        y_guide_test,
        guide_mean_probs,
        guide_uncertainty,
        guide_test[GRNA_COL].to_numpy()
    ),
    (
        'Familiar guide, cross assay',
        y_change_test[familiar_mask],
        change_mean_probs[familiar_mask],
        change_uncertainty[familiar_mask],
        change_test.loc[familiar_mask, GRNA_COL].to_numpy()
    ),
    (
        'Novel guide, cross assay',
        y_change_test[novel_mask],
        change_mean_probs[novel_mask],
        change_uncertainty[novel_mask],
        change_test.loc[novel_mask, GRNA_COL].to_numpy()
    )
]

rows = []

for name, y, p, u, guides in conditions:
    prevalence = y.mean()

    rows.append({
        'condition': name,
        'guides': len(np.unique(guides)),
        'rows': len(y),
        'positives': int(y.sum()),
        'prevalence': prevalence,
        'mean prediction': p.mean(),
        'prediction / prevalence': p.mean() / prevalence,
        'AUROC': roc_auc_score(y, p),
        'AUPRC': average_precision_score(y, p),
        'Brier': brier_score_loss(y, p),
        'ECE': ece(y, p),
        'mean uncertainty': u.mean(),
        '95th pct uncertainty': np.quantile(u, 0.95)
    })

condition_results = pd.DataFrame(rows)

condition_results.style.format({
    'prevalence': '{:.6f}',
    'mean prediction': '{:.6f}',
    'prediction / prevalence': '{:.3f}',
    'AUROC': '{:.4f}',
    'AUPRC': '{:.4f}',
    'Brier': '{:.6f}',
    'ECE': '{:.6f}',
    'mean uncertainty': '{:.2e}',
    '95th pct uncertainty': '{:.2e}'
})

,condition,guides,rows,positives,prevalence,mean prediction,prediction / prevalence,AUROC,AUPRC,Brier,ECE,mean uncertainty,95th pct uncertainty
0,"Novel guide, same assay",12,248614,90,0.000362,0.000391,1.080,0.8577,0.0117,0.000361,0.000235,8.64e-08,3.83e-07
1,"Familiar guide, cross assay",12,248375,2528,0.010178,0.000345,0.034,0.8885,0.1004,0.010155,0.009833,5.82e-08,2.23e-07
2,"Novel guide, cross assay",10,272616,4082,0.014973,0.000487,0.033,0.8334,0.1069,0.014935,0.014487,1.63e-07,8.59e-07


### Results

Cross-assay evaluation produced a clear separation between ranking performance and probability calibration. On guide-familiar CHANGE-seq examples, the ensemble achieved an AUROC of 0.8885, exceeding the 0.8577 obtained on the held-out GUIDE-seq test set. However, the mean predicted probability was only 0.000345 compared with an observed prevalence of 0.010178, producing an ECE of 0.009833. This shows that strong discrimination was retained despite substantial probability miscalibration.

Performance on guide-novel CHANGE-seq examples was weaker, with AUROC decreasing to 0.8334. These examples also exhibited greater ensemble disagreement than either the GUIDE-seq test set or the guide-familiar CHANGE-seq subset. This suggests that unfamiliar gRNAs contribute an additional generalisation challenge beyond the assay shift itself.

Importantly, ensemble disagreement did not increase on the guide-familiar cross-assay subset despite its severe calibration deterioration. This indicates that an assay-dependent shift shared across ensemble members may not necessarily be reflected by increased epistemic disagreement.

## Label Distribution Shift

Before matching individual genomic sites, the overall frequency of positive labels is compared across assays. Because pooled prevalence can be affected by differences in guide composition, both pooled prevalence and the distribution of per-guide positive rates are reported.